In [4]:
#First attempted to scrape information from AWS own produced graphics. Pivoted to then using nfl_data_py to scrape.
#
import pandas as pd
import nfl_data_py as nfl
import plotly.graph_objects as go
import numpy as np
from datetime import datetime


df = nfl.import_pbp_data(years=[2025])
bo_nix_passes = df[(df['posteam'] == 'DEN') & (df['passer_player_name'] == 'B.Nix') & (df['pass_attempt'] == 1)].copy()
df.columns.tolist()
viz_cols = [
    'week', 'qtr', 'down', 'ydstogo', 'game_date',
    'passer_player_name', 'receiver_player_name',
    'pass_attempt', 'complete_pass', 'pass_location', 'pass_length',
    'air_yards', 'yards_after_catch', 'yards_gained',
    'yardline_100', 'side_of_field',
    'desc', 'touchdown', 'interception', 'incomplete_pass',
    'score_differential', 'posteam_score', 'defteam_score'
]
bo_nix_visual = bo_nix_passes[viz_cols].copy()

def target_position(row):
    y_positions= {
        'left': 13.25,
        'middle': 26.25, 
        'right': 40.05,
    }
    y = y_positions.get(row['pass_location'], 26.65)
    if pd.notna(row['air_yards']):
        x = row['yardline_100']-row['air_yards']
    else:
        x= row['yardline_100']
    x = max(0, min(100,x))
    return x,y
bo_nix_passes[['target_x', 'target_y']] = bo_nix_passes.apply(lambda row: pd.Series(target_position(row)), axis = 1)

def create_football_field():
    fig = go.Figure()

    fig.add_shape(
        type = "rect",
        x0 = 0, y0 = 0, x1 = 100, y1 = 53.3,
        fillcolor = "#000000",
        line = dict(width = 0),
        layer = "below"
    )
    fig.add_shape(
        type="rect", 
        x0=0, y0=0, x1=100, y1=53.3,
        line=dict(color="white", width=3), 
        fillcolor="rgba(0,0,0,0)"
    )
    for yard in range(10, 100, 10):
        fig.add_shape(
            type="line",
            x0=yard, y0=0, x1=yard, y1=53.3,
            line=dict(color="white", width=2)
        )
        yard_label = yard if yard <= 50 else 100 - yard
        fig.add_annotation(
            x=yard, y=3,
            text=str(yard_label),
            showarrow=False,
            font=dict(color="white", size=12, family="Arial Black")
        )
        fig.add_annotation(
            x=yard, y=50.3,
            text=str(yard_label),
            showarrow=False,
            font=dict(color="white", size=12, family="Arial Black")
        )
    fig.add_shape(
        type = "line",
        x0 = 50, y0 = 0, x1 = 50, 
        line = dict(color = "white", width = 2)
    )
    for yard in range (0,101):
        fig.add_shape(
            type = "line",
            x0=yard, y0=18.5, x1=yard, y1=19,
            line=dict(color="white", width=1)
        )
        fig.add_shape(
            type="line",
            x0=yard, y0=34.3, x1=yard, y1=34.8,
            line=dict(color="white", width=1)
        )
    fig.add_shape(type="line", x0=10, y0=0, x1=10, y1=53.3,line=dict(color="white", width=3))
    
    fig.add_shape(type="line", x0=90, y0=0, x1=90, y1=53.3,line=dict(color="white", width=3))
    
    fig.add_shape(type="line", x0=0, y0=0, x1=0, y1=53.3, line=dict(color="white", width=4))
    
    fig.add_shape(type="line", x0=100, y0=0, x1=100, y1=53.3, line=dict(color="white", width=4))
    
    return fig

def add_jitter(x_series, y_series, jitter=1.2):
    df = pd.DataFrame({'x': x_series, 'y': y_series})
    new_x, new_y = [], []

    for (x, y), group in df.groupby(['x', 'y']):
        n = len(group)
        if n == 1:
            new_x.append(x)
            new_y.append(y)
        else:
            angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
            radii = np.random.uniform(0.3 * jitter, jitter, n)
            new_x.extend(x + np.cos(angles) * radii)
            new_y.extend(y + np.sin(angles) * radii)

    jittered = pd.DataFrame({'x': new_x, 'y': new_y})
    return jittered['x'].tolist(), jittered['y'].tolist()

def create_passing_chart(plays_df, title = "Bo Nix 2025 Passing Chart"):
    fig = create_football_field()
    plays_df = plays_df.copy()
    plays_df['plot_x'], plays_df['plot_y'] = add_jitter(plays_df['target_x'], plays_df['target_y'])
    
    touchdowns = plays_df[plays_df['touchdown'] == 1].copy()
    interceptions = plays_df[plays_df['interception'] == 1].copy()
    complete = plays_df[
        (plays_df['complete_pass'] == 1) & 
        (plays_df['touchdown'] == 0)].copy()
    incomplete = plays_df[
        (plays_df['complete_pass'] == 0) & 
        (plays_df['interception'] == 0)].copy()

    # Add touchdowns
    if len(touchdowns) > 0:
        fig.add_trace(go.Scatter(
            x=touchdowns['plot_x'],
            y=touchdowns['plot_y'],
            mode='markers',
            name='Touchdown',
            marker=dict(
                size=16,
                color='#0000FF',
                line=dict(width=2, color='white'),
                opacity=0.9
            ),
            customdata=touchdowns[['yardline_100', 'receiver_player_name', 'week', 'qtr', 'down', 'ydstogo', 'yards_gained', 'air_yards', 
                                  'pass_location', 'desc']].values,
            hovertemplate=(
                '<b>🏈 TOUCHDOWN to %{customdata[1]}!</b><br>'
                '<b>Week %{customdata[2]:.0f}</b> - Q%{customdata[3]:.0f}<br>'
                '<b>Down & Distance:</b> %{customdata[4]:.0f} & %{customdata[5]:.0f}<br>'
                '<b>Yards Gained:</b> %{customdata[6]:.0f}<br>'
                '<b>Air Yards:</b> %{customdata[7]:.0f}<br>'
                '<b>Pass Location:</b> %{customdata[8]}<br>'
                '<b>Line of Scrimmage:</b> %{customdata[0]:.0f} yards to endzone<br>'
                '<extra></extra>'
            )
        ))
    
    # Add interceptions
    if len(interceptions) > 0:
        fig.add_trace(go.Scatter(
            x=interceptions['plot_x'],
            y=interceptions['plot_y'],
            mode='markers',
            name='Interception',
            marker=dict(
                size=16,
                color='#FF0000',
                line=dict(width=2, color='white'),
                opacity=0.9
            ),
            customdata=interceptions[['yardline_100', 'receiver_player_name', 'week', 'qtr','down', 'ydstogo', 'air_yards', 
                                      'pass_location', 'desc']].values,
            hovertemplate=(
                '<b>🚫 INTERCEPTION (Target: %{customdata[1]})</b><br>'
                '<b>Week %{customdata[2]:.0f}</b> - Q%{customdata[3]:.0f}<br>'
                '<b>Down & Distance:</b> %{customdata[4]:.0f} & %{customdata[5]:.0f}<br>'
                '<b>Air Yards:</b> %{customdata[6]:.0f}<br>'
                '<b>Pass Location:</b> %{customdata[7]}<br>'
                '<b>Line of Scrimmage:</b> %{customdata[0]:.0f} yards to endzone<br>'
                '<extra></extra>'
            )
        ))

    # Add completions
    if len(complete) > 0:
        fig.add_trace(go.Scatter(
            x=complete['plot_x'],
            y=complete['plot_y'],
            mode='markers',
            name='Complete',
            marker=dict(
                size=14,
                color='#00ff00',
                line=dict(width=2, color='white'),
                opacity=0.8
            ),
            text=complete.apply(lambda row:
                f"<b>✅ Complete to {row['receiver_player_name']}</b><br>"
                f"<b>Week {int(row['week'])}</b> - Q{int(row['qtr'])}<br>"
                f"<b>Down & Distance:</b> {int(row['down'])} & {int(row['ydstogo'])}<br>"
                f"<b>Yards Gained:</b> {int(row['yards_gained'])}<br>"
                f"<b>Air Yards:</b> {int(row['air_yards']) if pd.notna(row['air_yards']) else 'N/A'}<br>"
                f"<b>Pass Location:</b> {row['pass_location'].title() if pd.notna(row['pass_location']) else 'N/A'}<br>"
                f"<b>Field Position:</b> {int(row['yardline_100'])} yards to endzone<br>",
                axis=1
            ),
            hovertemplate='%{text}<extra></extra>'
        ))

    # Add incomplete passes
    if len(incomplete) > 0:
        fig.add_trace(go.Scatter(
            x=incomplete['plot_x'],
            y=incomplete['plot_y'],
            mode='markers',
            name='Incomplete',
            marker=dict(
                size=14,
                color='white',
                line=dict(width=2, color='#888'),
                opacity=0.7
            ),
            text=incomplete.apply(lambda row:
                f"<b>❌ Incomplete to {row['receiver_player_name']}</b><br>"
                f"<b>Week {int(row['week'])}</b> - Q{int(row['qtr'])}<br>"
                f"<b>Down & Distance:</b> {int(row['down'])} & {int(row['ydstogo'])}<br>"
                f"<b>Air Yards:</b> {int(row['air_yards']) if pd.notna(row['air_yards']) else 'N/A'}<br>"
                f"<b>Pass Location:</b> {row['pass_location'].title() if pd.notna(row['pass_location']) else 'N/A'}<br>"
                f"<b>Field Position:</b> {int(row['yardline_100'])} yards to endzone<br>",
                axis=1
            ),
            hovertemplate='%{text}<extra></extra>'
        ))
    
    # Layout and stats (rest of your code is fine)
    total_attempts = len(plays_df)
    completions = int(plays_df['complete_pass'].sum())
    comp_pct = (completions / total_attempts * 100) if total_attempts > 0 else 0
    weeks = sorted(plays_df['week'].unique())
    total_yards = int(plays_df['yards_gained'].sum())
    touchdowns_count = int(plays_df['touchdown'].sum())
    interceptions_count = int(plays_df['interception'].sum())

    fig.update_layout(
        title={
            'text': (f"{title}<br>"
                    f"<sub>Weeks {min(weeks)}-{max(weeks)} | "
                    f"{completions}/{total_attempts} ({comp_pct:.1f}%) | "
                    f"{total_yards} Yards | {touchdowns_count} TDs | {interceptions_count} INTs</sub>"),
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 24, 'color': 'white', 'family': 'Arial Black'}
        },
        xaxis=dict(range=[-2, 102], showgrid=False, zeroline=False, showticklabels=False, title=""),
        yaxis=dict(range=[-3, 55.3], showgrid=False, zeroline=False, showticklabels=False,
                   scaleanchor="x", scaleratio=1, title=""),
        plot_bgcolor='#1a1a1a',
        paper_bgcolor='#1a1a1a',
        font=dict(color='white'),
        height=750,
        hovermode='closest',
        legend=dict(x=0.02, y=0.98, bgcolor='rgba(0,0,0,0.7)', bordercolor='white', borderwidth=2, font=dict(size=14))
    )
    
    return fig

fig = create_passing_chart(bo_nix_passes, "Bo Nix Passing Chart - 2025 Season")

fig.write_html('bo_nix_passing_chart_2025.html')
print("\n✅ Visualization saved as 'bo_nix_passing_chart_2025.html'")
print("📂 Open the file in your browser to view!")

print("\n" + "="*60)
print("BO NIX PASSING STATS - 2025 SEASON")
print("="*60)
print(f"Total Attempts: {len(bo_nix_passes)}")
print(f"Completions: {int(bo_nix_passes['complete_pass'].sum())}")
print(f"Completion %: {(bo_nix_passes['complete_pass'].mean() * 100):.1f}%")
print(f"Total Yards: {int(bo_nix_passes['yards_gained'].sum())}")
print(f"Yards/Attempt: {bo_nix_passes['yards_gained'].mean():.1f}")
print(f"Touchdowns: {int(bo_nix_passes['touchdown'].sum())}")
print(f"Interceptions: {int(bo_nix_passes['interception'].sum())}")
print(f"Weeks: {sorted(bo_nix_passes['week'].unique())}")

print("\n🎯 Top 5 Targets:")
print(bo_nix_passes['receiver_player_name'].value_counts().head())

print("\n📍 Pass Location Distribution:")
print(bo_nix_passes['pass_location'].value_counts())

# Create separate chart for each week
weeks = sorted(bo_nix_passes['week'].unique())

for week in range(1, 18):
    week_data = bo_nix_passes[bo_nix_passes['week'] == week]
    if len(week_data) > 0:
        fig = create_passing_chart(week_data, f"Bo Nix Passing Chart - Week {week}")  # Fixed!
        fig.write_html(f'bo_nix_week_{week}_2024.html')
        print(f"✅ Created Week {week} visualization ({len(week_data)} passes)")

print("All visualizations complete!")

ModuleNotFoundError: No module named 'nfl_data_py'